# Noise and mixed states
**Quantum Computing · Week 6 (Noise and real hardware) · Day 1**

Lesson page: https://sara-aminpour.com/tutorials/qc/w6d1.html

In [ ]:
# On Google Colab, this installs qiskit and qiskit-aer (about a minute). On your own computer, skip it and
# install the packages once instead: pip install qiskit==2.5.2 qiskit-aer==0.17.2 scikit-learn torch matplotlib
import sys
if "google.colab" in sys.modules:
    %pip install --quiet qiskit==2.5.2 qiskit-aer==0.17.2

Real qubits are noisy. A noisy qubit is no longer a definite $|\psi\rangle$ but a **mixed state**,
and its Bloch arrow shrinks *inside* the sphere. Today: three kinds of noise, and a noisy simulator.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from qiskit.quantum_info import Statevector, DensityMatrix, Pauli, Kraus

plus = DensityMatrix(Statevector.from_label("+"))           # a pure state, written as a density matrix
mixed = DensityMatrix(np.eye(2) / 2)                        # a fair coin we know nothing more about

def bloch(rho):
    return [round(float(np.real(rho.expectation_value(Pauli(p)))), 3) + 0.0 for p in "XYZ"]

for name, rho in [("|+>  (pure)", plus), ("I/2 (mixed)", mixed)]:
    print(name, "  probabilities of 0 and 1:", rho.probabilities().round(3), "  Bloch vector:", bloch(rho))

Same 50/50 statistics in the 0/1 basis, yet completely different states: $|+\rangle$ has an arrow of
length 1 pointing along $x$, the mixed state an arrow of length 0. (Tomorrow is all about these matrices.)

## Three kinds of noise
Each noise channel is a recipe of **Kraus operators** $K_i$: $\rho \to \sum_i K_i \rho K_i^\dagger$.
Watch what each does to the Bloch vector of a state tilted away from the poles.

In [ ]:
def depolarizing(p):
    return Kraus([np.sqrt(1 - 3 * p / 4) * np.eye(2)] + [np.sqrt(p / 4) * Pauli(s).to_matrix() for s in "XYZ"])

def dephasing(p):                           # a Z error with probability p/2
    return Kraus([np.sqrt(1 - p / 2) * np.eye(2), np.sqrt(p / 2) * Pauli("Z").to_matrix()])

def amplitude_damping(g):                   # |1> decays to |0> with probability g
    return Kraus([np.array([[1, 0], [0, np.sqrt(1 - g)]]), np.array([[0, np.sqrt(g)], [0, 0]])])

start = DensityMatrix(Statevector([np.cos(1.0), np.exp(0.6j) * np.sin(1.0)]))
print(f"{'clean state:':28s}", bloch(start))
for name, channel in [("depolarizing", depolarizing), ("dephasing", dephasing), ("amplitude damping", amplitude_damping)]:
    print(f"{name + ', p = 0.5:':28s}", bloch(start.evolve(channel(0.5))))

Depolarizing shrinks the whole arrow, dephasing shrinks only its $x$ and $y$ parts, and amplitude
damping pulls everything towards $|0\rangle$, the north pole.

## Noise over time: T1 and T2
On hardware, noise grows with time. Energy relaxation (amplitude damping) has a time constant $T_1$:
a qubit left in $|1\rangle$ is still there with probability $e^{-t/T_1}$. Dephasing has a time constant
$T_2$: the arrow's $x$ and $y$ parts fade like $e^{-t/T_2}$.

In [ ]:
T1, T2 = 100e-6, 80e-6                      # seconds: typical orders of magnitude for superconducting qubits
times = np.linspace(0, 300e-6, 61)
p_one = [DensityMatrix.from_label("1").evolve(amplitude_damping(1 - np.exp(-t / T1))).probabilities()[1] for t in times]
coherence = [bloch(plus.evolve(dephasing(1 - np.exp(-t / T2))))[0] for t in times]

plt.plot(times * 1e6, p_one, color="#ff8a50", lw=2.5, label=r"$P(1)$ after preparing $|1\rangle$ (T1)")
plt.plot(times * 1e6, coherence, color="#7b5cfa", lw=2.5, label=r"$\langle X\rangle$ after preparing $|+\rangle$ (T2)")
plt.xlabel("waiting time (microseconds)")
plt.ylabel("what's left")
plt.legend()
plt.show()

## A noisy simulation with Qiskit Aer
`qiskit_aer` can attach a noise model to a simulator. Here every gate gets a little depolarizing noise,
and the Bell state's perfect correlations start to leak.

In [ ]:
from qiskit import QuantumCircuit, transpile
from qiskit_aer import AerSimulator
from qiskit_aer.noise import NoiseModel, depolarizing_error

bell = QuantumCircuit(2)
bell.h(0)
bell.cx(0, 1)
bell.measure_all()

noise = NoiseModel()
noise.add_all_qubit_quantum_error(depolarizing_error(0.05, 1), ["h"])
noise.add_all_qubit_quantum_error(depolarizing_error(0.10, 2), ["cx"])

for name, sim in [("ideal", AerSimulator()), ("noisy", AerSimulator(noise_model=noise))]:
    counts = sim.run(transpile(bell, sim), shots=4000, seed_simulator=7).result().get_counts()
    print(f"{name:>5}: {dict(sorted(counts.items()))}")

## Your turn: bigger circuits, more noise
Build GHZ states on $n = 2, \dots, 7$ qubits (H, then a chain of CNOTs) and run them on a simulator
whose CNOTs have a 2% depolarizing error. Plot the share of shots that give one of the two correct
outcomes, all 0s or all 1s, against $n$.

In [ ]:
noise2 = NoiseModel()
noise2.add_all_qubit_quantum_error(depolarizing_error(0.02, 2), ["cx"])
sim = AerSimulator(noise_model=noise2)
good = []
for n in range(2, 8):
    ghz = QuantumCircuit(n)
    # TODO: H on qubit 0, then CNOTs 0->1, 1->2, ..., then measure_all()
    good.append(0.0)    # TODO: (counts of "0"*n + counts of "1"*n) / shots
print(good)

---
## Solution
Try the exercise yourself first: it's fine to get stuck for a minute. Then compare with this one (there are many good answers).

In [ ]:
noise2 = NoiseModel()
noise2.add_all_qubit_quantum_error(depolarizing_error(0.02, 2), ["cx"])
sim = AerSimulator(noise_model=noise2)
ns, good = range(2, 8), []
for n in ns:
    ghz = QuantumCircuit(n)
    ghz.h(0)
    for q in range(n - 1):
        ghz.cx(q, q + 1)
    ghz.measure_all()
    counts = sim.run(transpile(ghz, sim), shots=4000, seed_simulator=1).result().get_counts()
    good.append((counts.get("0" * n, 0) + counts.get("1" * n, 0)) / 4000)
    print(f"n = {n}: {n - 1} CNOTs, correct outcomes {good[-1]:.3f}")
plt.plot(list(ns), good, "o-", color="#7b5cfa")
plt.xlabel("number of qubits n")
plt.ylabel("share of correct outcomes")
plt.title("Every extra CNOT lets in a little more noise")
plt.show()